# 13 – Favs

Esplorazione e data cleaning del dataset `favs.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `username` | Nome utente MAL |
| `fav_type` | Tipo di preferito (`anime`, `character`, `people`, `company`) |
| `id` | ID dell'elemento preferito (MAL ID relativo al tipo) |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_favs = pd.read_csv('../datasets/favs.csv')
print(f'Shape: {df_favs.shape}')
print()
df_favs.info()
df_favs.head()

Shape: (4178747, 3)

<class 'pandas.DataFrame'>
RangeIndex: 4178747 entries, 0 to 4178746
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   username  str  
 1   fav_type  str  
 2   id        int64
dtypes: int64(1), str(2)
memory usage: 95.6 MB


,username,fav_type,id
0,ishikawas,anime,45649
1,ishikawas,anime,38680
2,ishikawas,anime,795
3,ishikawas,anime,37510
4,ishikawas,anime,820


Il dataset contiene **4.178.747 righe** e **3 colonne**. I tipi di dati sono adeguati: `int64` per gli ID e `str` per le colonne testuali.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [2]:
n_originale = len(df_favs)

mask_dup = df_favs.duplicated(keep=False)       
n_righe_coinvolte = mask_dup.sum()               
n_gruppi = df_favs[mask_dup].duplicated(keep='first').sum()  
n_tenute = n_righe_coinvolte - n_gruppi         

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_favs.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_favs):,}')

Righe totali coinvolte in duplicazioni : 0
  → prime occorrenze mantenute         : 0
  → occorrenze extra rimosse           : 0

Righe prima della rimozione : 4,178,747
Righe dopo la rimozione     : 4,178,747


Nessun duplicato esatto trovato. Tutte le 4.178.747 righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `username`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria di `profiles.csv`. Per una chiave esterna le statistiche descrittive non hanno significato interpretativo.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni username presente qui deve esistere in `profiles_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

I valori duplicati sono **attesi**: uno stesso utente può avere più preferiti.

In [3]:
df_profiles = pd.read_csv('../datasets_cleaned/profiles_clean.csv')
df_favs['username'] = df_favs['username'].str.strip()
mask_orphan = check_fk(df_favs['username'], df_profiles['username'], child_df=df_favs)
print(f'Null in username              : {df_favs["username"].isna().sum()}')
print(f'Duplicati in username (attesi): {df_favs["username"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        username
  Colonna PK  (tabella padre)         username
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       4,178,747
  Valori null nella FK                4  (0.00%)
  Valori non-null nella FK            4,178,743  (100.00%)
  Valori unici nella PK padre         335,476

  ✓  Righe con FK valida              4,158,598  (99.52%)
  ✗  Righe orfane (FK non in PK)      20,145  (0.48%)
     → ID orfani unici                1,188

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

20,145 riga/e (0.48%) violano l'integrità referenziale.

Campione righe orfane (prime 10)
────────────────────────────────────────────────────────────────────────────────

    username   fav_type      id
0  ishikawas      anime   45649
1  ishika

**Osservazioni:**
- Ci sono 4 valori nulli in `username` da rimuovere.
- **Integrità referenziale**: 20.145 righe orfane (0.48%) con 1.188 username unici non presenti in `profiles_clean.csv`, anche queste da rimuovere.

In [4]:
# Rimozione null
print(f'Null in username prima della pulizia: {df_favs["username"].isna().sum()}')
df_favs.dropna(subset=['username'], inplace=True)
print(f'Null in username dopo la pulizia    : {df_favs["username"].isna().sum()}')

# Rimozione righe orfane
if mask_orphan.any():
    n_orfane = mask_orphan.sum()
    df_favs = df_favs[~mask_orphan].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane}')
else:
    print('Nessuna riga orfana da rimuovere.')

print(f'Righe dopo pulizia username: {len(df_favs):,}')

Null in username prima della pulizia: 4
Null in username dopo la pulizia    : 0
Righe orfane rimosse : 20145
Righe dopo pulizia username: 4,158,598


### 2.2 `fav_type`

In [5]:
df_favs['fav_type'] = df_favs['fav_type'].str.strip()
analyze(df_favs['fav_type'])


  Nome serie                     fav_type
  dtype                          str
  Dimensione                     4,158,598
  Range indice                   0 … 4158597
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   4,158,598
  Valori non nulli               4,158,598  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   4  (0.00%)
  Valori duplicati               4,158,594  (100.00%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  5  caratteri
  Lunghezza max                  9  caratteri
  Lunghezza media                6.83  caratteri
  Lunghezza mediana              6.0000  caratteri
  Dev. standard lunghezza        1.78
  IQR lunghezza                  4.0000

Valor

**Osservazioni:**
- Nessun null. I quattro valori (`anime`, `character`, `people`, `company`) sono coerenti con le categorie di preferiti di MAL.

**Nessuna pulizia necessaria**.

### 2.3 `id`

Rappresenta l'ID dell'elemento preferito.

In [6]:
analyze(df_favs['id'])


  Nome serie                     id
  dtype                          int64
  Dimensione                     4,158,598
  Range indice                   0 … 4158597
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   4,158,598
  Valori non nulli               4,158,598  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           0  (0.00%)
  Positivi                       4,158,598   (100.00%)
  Negativi                       0   (0.00%)
  Valori unici                   51,478  (1.24%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            1
  Max                            281,988
  Range                          281,987
  Media                          35,560.00
  Mediana                        12,115.00
  Moda/e                         21
  Dev. standar

- Nessun null, dtype già `int64`.
- I duplicati sono attesi in quanto lo stesso ID può essere preferito da utenti diversi, e lo stesso ID può comparire con `fav_type` diversi.
- Essendo una colonna **contestuale** (il dataset padre dipende da `fav_type`), effettuiamo il controllo di integrità referenziale separatamente.

Per `fav_type = 'company'` non esiste un dataset padre disponibile. Per gli altri tre tipi effettuiamo il controllo di integrità referenziale separatamente:
- `anime` → `details_clean.csv` (`mal_id`)
- `character` → `characters_clean.csv` (`character_mal_id`)
- `people` → `person_details_clean.csv` (`person_mal_id`)

In [7]:
df_details    = pd.read_csv('../datasets_cleaned/details_clean.csv',        usecols=['mal_id'])
df_characters = pd.read_csv('../datasets_cleaned/characters_clean.csv',     usecols=['character_mal_id'])
df_persons    = pd.read_csv('../datasets_cleaned/person_details_clean.csv', usecols=['person_mal_id'])

subset_anime = df_favs[df_favs['fav_type'] == 'anime'].copy()
print(f"fav_type = 'anime' ({len(subset_anime):,} righe)")
check_fk(subset_anime['id'], df_details['mal_id'], child_df=subset_anime)
print()

subset_character = df_favs[df_favs['fav_type'] == 'character'].copy()
print(f"fav_type = 'character' ({len(subset_character):,} righe)")
check_fk(subset_character['id'], df_characters['character_mal_id'], child_df=subset_character)
print()

subset_people = df_favs[df_favs['fav_type'] == 'people'].copy()
print(f"fav_type = 'people' ({len(subset_people):,} righe)")
check_fk(subset_people['id'], df_persons['person_mal_id'], child_df=subset_people)


fav_type = 'anime' (1,524,500 righe)

  Colonna FK  (tabella figlia)        id
  Colonna PK  (tabella padre)         mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       1,524,500
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            1,524,500  (100.00%)
  Valori unici nella PK padre         28,955

  ✓  Righe con FK valida              1,519,468  (99.67%)
  ✗  Righe orfane (FK non in PK)      5,032  (0.33%)
     → ID orfani unici                39

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

5,032 riga/e (0.33%) violano l'integrità referenziale.

Campione righe orfane (prime 10)
────────────────────────────────────────────────────────────────────────────────

           username fav_type     id
243   -----Ren

11         False
12         False
13         False
14         False
34         False
           ...  
4158573    False
4158574    False
4158575    False
4158576    False
4158577    False
Name: id, Length: 857968, dtype: bool

**Osservazioni integrità referenziale per tipo:**
- Per il caso fav_type = 'anime' ci sono 5,032 righe orfane. 
- Per il caso fav_type = 'character' ci sono 20 righe orfane. 
- Per il caso fav_type = 'people' ci sono 13 righe orfane. 

In totale ci sono 5,065 righe orfane da rimuovere. 

In [8]:
righe_prima = len(df_favs)

valid_anime   = df_details['mal_id']
valid_char    = df_characters['character_mal_id']
valid_people  = df_persons['person_mal_id']

mask_anime   = (df_favs['fav_type'] == 'anime')     & ~df_favs['id'].isin(valid_anime)
mask_char    = (df_favs['fav_type'] == 'character') & ~df_favs['id'].isin(valid_char)
mask_people  = (df_favs['fav_type'] == 'people')    & ~df_favs['id'].isin(valid_people)

mask_orfane  = mask_anime | mask_char | mask_people

print(f"fav_type='anime'    : {mask_anime.sum():,} righe orfane rimosse")
print(f"fav_type='character': {mask_char.sum():,} righe orfane rimosse")
print(f"fav_type='people'   : {mask_people.sum():,} righe orfane rimosse")

df_favs = df_favs[~mask_orfane].reset_index(drop=True)
print(f'\nRighe prima : {righe_prima:,}')
print(f'Righe dopo  : {len(df_favs):,}')


fav_type='anime'    : 5,032 righe orfane rimosse
fav_type='character': 20 righe orfane rimosse
fav_type='people'   : 13 righe orfane rimosse

Righe prima : 4,158,598
Righe dopo  : 4,153,533


### 2.4 Chiave primaria `(username, fav_type, id)`

La chiave primaria è la tripla `(username, fav_type, id)`: verifichiamo che sia univoca dopo la pulizia.

In [9]:
n_pk_dup = df_favs.duplicated(subset=['username', 'fav_type', 'id'], keep=False).sum()
print(f'Duplicati su (username, fav_type, id): {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_favs.drop_duplicates(subset=['username', 'fav_type', 'id'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_favs):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su (username, fav_type, id): 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [10]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_favs):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_favs):>10,}')
print()

df_favs.to_csv('../datasets_cleaned/favs_clean.csv', index=False)
print('Salvato: datasets_cleaned/favs_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :  4,178,747
Righe dopo cleaning  :  4,153,533
Righe rimosse totali :     25,214

Salvato: datasets_cleaned/favs_clean.csv
